# ASPPNeXt Fine-Tuning for Maximum Accuracy and Loss Convergence
This notebook is tailored for Kaggle (16GB GPU).
- Strong augmentation
- Cosine annealing LR scheduler
- Large batch size (adjust as needed)
- High model capacity
- Combined Focal Tversky + Dice loss
- No early stopping, allow overfitting

**Edit batch size/model dims in the config cell if you get OOM!**

In [ ]:
# --- Imports and Environment Setup ---
import torch as T
import torch.nn as nn
import torch.nn.functional as F
import os
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
from torchvision import transforms
from tqdm.notebook import tqdm
from torch.cuda.amp import GradScaler, autocast
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
T.backends.cudnn.benchmark = True
device = T.device('cuda' if T.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# --- Configurations ---
MODELS_DIR = '/kaggle/working/ASPPNeXt_models'
os.makedirs(MODELS_DIR, exist_ok=True)
# You can lower batch_size/base_dim if you hit OOM errors
BATCH_SIZE = 12   # 8-12 is reasonable for 16GB GPU
BASE_DIM = 48     # Increase for more capacity (default 32/48)
NUM_EPOCHS = 50


In [ ]:
# --- Strong Data Augmentation for Overfitting-Resistant Training ---
strong_aug = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomResizedCrop((384, 512), scale=(0.8, 1.0)),
    transforms.GaussianBlur(3, sigma=(0.1, 2.0)),
    transforms.ToTensor()
])


In [ ]:
# --- Dataset & Dataloader (adapted to use strong_aug) ---
class ASPPNeXtDataset(Dataset):
    def __init__(self, image_dir, depth_dir, mask_dir, window_size=8, transform=None):
        self.window_size = window_size
        self.transform = transform
        for d in [image_dir, depth_dir, mask_dir]:
            if not os.path.isdir(d):
                raise FileNotFoundError(f'Directory {d} does not exist')
        self.image_files = sorted([f for f in os.listdir(image_dir) if f.endswith('.jpg')],
                                key=lambda x: int(x.split('_')[0]) if '_' in x else int(x.split('.')[0]))
        self.depth_files = [f.replace('.jpg', '.png').replace('image', 'image_depth') for f in self.image_files]
        self.mask_files = [f.replace('.jpg', '.png') for f in self.image_files]
        self.image_paths = [os.path.join(image_dir, f) for f in self.image_files]
        self.depth_paths = [os.path.join(depth_dir, f) for f in self.depth_files]
        self.mask_paths = [os.path.join(mask_dir, f) for f in self.mask_files]
        for p in self.image_paths + self.depth_paths + self.mask_paths:
            if not os.path.exists(p):
                raise FileNotFoundError(f'File {p} does not exist')
    def pad_to_square_multiple(self, arr, multiple=8):
        h, w = arr.shape[:2]
        max_dim = max(h, w)
        final_dim = ((max_dim + multiple - 1) // multiple) * multiple
        pad_h = final_dim - h
        pad_w = final_dim - w
        pad_top = pad_h // 2
        pad_bottom = pad_h - pad_top
        pad_left = pad_w // 2
        pad_right = pad_w - pad_left
        if arr.ndim == 3:
            return np.pad(arr, ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)), mode='constant')
        else:
            return np.pad(arr, ((pad_top, pad_bottom), (pad_left, pad_right)), mode='constant')
    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        if img is None: raise ValueError(f'Failed to read image: {self.image_paths[idx]}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        depth = cv2.imread(self.depth_paths[idx], cv2.IMREAD_GRAYSCALE)
        if depth is None: raise ValueError(f'Failed to read depth: {self.depth_paths[idx]}')
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        if mask is None: raise ValueError(f'Failed to read mask: {self.mask_paths[idx]}')
        img = self.pad_to_square_multiple(img, self.window_size)
        depth = self.pad_to_square_multiple(depth, self.window_size)
        mask = self.pad_to_square_multiple(mask, self.window_size)
        # --- Apply strong augmentation to training only ---
        if self.transform:
            img = self.transform(img)
        else:
            img = T.from_numpy(img).float().permute(2, 0, 1) / 255.0
        depth = T.from_numpy(depth).float().unsqueeze(0) / 255.0
        mask = T.from_numpy(mask).long()
        return img, depth, mask
    def __len__(self):
        return len(self.image_paths)


In [ ]:
# --- Dataloader Setup ---
base_img = '/kaggle/input/cwf-788/IMAGE512x384'
base_depth = '/kaggle/input/cwf-788/Depth_Features'
base_mask = '/kaggle/input/cwf-788/Weed_Masks'
splits = {
    'train': ('train_new', 'train_new', 'train'),
    'val': ('validation_new', 'validation_new', 'val'),
    'test': ('test_new', 'test_new', 'test'),
}
model_window_sizes = (8, 8, 4, 2)
window_size = max(model_window_sizes)
dataloaders = {}
for phase, (img_s, depth_s, mask_s) in splits.items():
    img_dir = os.path.join(base_img, img_s)
    depth_dir = os.path.join(base_depth, depth_s)
    mask_dir = os.path.join(base_mask, mask_s)
    transform = strong_aug if phase == 'train' else None
    ds = ASPPNeXtDataset(img_dir, depth_dir, mask_dir, window_size, transform=transform)
    dataloaders[phase] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=(phase=='train'), pin_memory=True, num_workers=2, prefetch_factor=2)
    print(f'{phase} dataset loaded with {len(ds)} samples')


In [ ]:
# --- Model Definition ---
# Paste your ASPPNeXtModel and all dependencies here (from your original notebook)
# ... [cut for brevity: user will paste all model classes here] ...


In [ ]:
# --- Model Instantiation ---
model = ASPPNeXtModel(
    in_ch_rgb=3,
    in_ch_depth=1,
    base_dim=BASE_DIM,
    num_heads=(2, 4, 8, 16),
    window_sizes=(8, 8, 4, 2),
    mlp_ratio=4,
    num_classes=2,
    output_upsample='dysample',
    use_feat_transform=True
).to(device)
print('Model parameters:', sum(p.numel() for p in model.parameters())//1e6, 'Million')


In [ ]:
# --- Optimizer & CosineAnnealingLR Scheduler ---
optimizer = T.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01, betas=(0.9, 0.999))
from torch.optim.lr_scheduler import CosineAnnealingLR
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-7)


In [ ]:
# --- Combined Loss: Focal Tversky + Dice ---
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
    def forward(self, preds, targets):
        targets_one_hot = F.one_hot(targets.clamp(0,1), num_classes=2).permute(0,3,1,2).float()
        probs = F.softmax(preds, dim=1)
        inter = (probs * targets_one_hot).sum((0,2,3))
        denom = (probs + targets_one_hot).sum((0,2,3))
        dice = (2*inter + self.smooth) / (denom + self.smooth)
        return 1 - dice.mean()
class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=0.75, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth
    def forward(self, preds, targets):
        targets_one_hot = F.one_hot(targets.clamp(0,1), num_classes=2).permute(0,3,1,2).float()
        probs = F.softmax(preds, dim=1)
        TP = (probs * targets_one_hot).sum((0,2,3))
        FP = (probs * (1-targets_one_hot)).sum((0,2,3))
        FN = ((1-probs) * targets_one_hot).sum((0,2,3))
        tversky = (TP + self.smooth) / (TP + self.alpha*FP + self.beta*FN + self.smooth)
        return T.mean(T.pow((1 - tversky), self.gamma))
class ComboLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super().__init__()
        self.ft = FocalTverskyLoss()
        self.dice = DiceLoss()
        self.alpha = alpha
    def forward(self, pred, target):
        return self.alpha * self.ft(pred, target) + (1 - self.alpha) * self.dice(pred, target)
loss_fn = ComboLoss(alpha=0.5).to(device)


In [ ]:
# --- Training Loop ---
scaler = GradScaler()
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loader = dataloaders['train']
    train_loss_sum = 0.0
    num_train_batches = len(train_loader)
    for img, depth, mask in tqdm(train_loader, desc=f'Train E{epoch}', leave=False):
        img, depth, mask = img.to(device), depth.to(device), mask.to(device)
        optimizer.zero_grad()
        with autocast():
            outputs = model(img, depth)
            loss = loss_fn(outputs, mask)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        T.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss_sum += loss.item()
        del img, depth, mask, outputs, loss
    scheduler.step()
    T.cuda.empty_cache()
    print(f'Epoch {epoch}: Train Loss: {train_loss_sum/num_train_batches:.4f}')
    # --- Validation (optional, add metric computation if desired) ---
    model.eval()
    val_loader = dataloaders['val']
    val_loss_sum = 0.0
    num_val_batches = len(val_loader)
    with T.no_grad():
        for img, depth, mask in tqdm(val_loader, desc=f'Val E{epoch}', leave=False):
            img, depth, mask = img.to(device), depth.to(device), mask.to(device)
            outputs = model(img, depth)
            loss = loss_fn(outputs, mask)
            val_loss_sum += loss.item()
            del img, depth, mask, outputs, loss
    T.cuda.empty_cache()
    print(f'Epoch {epoch}: Val Loss: {val_loss_sum/num_val_batches:.4f}')
    # Save best model based on val loss
    if epoch == 1 or val_loss_sum/num_val_batches < best_val_loss:
        best_val_loss = val_loss_sum/num_val_batches
        T.save(model.state_dict(), os.path.join(MODELS_DIR, 'best_model.pth'))
